In [ ]:
from typing import TypedDict, Literal
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from loguru import logger

load_dotenv(override=True)

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thingking": {
            "type": "disabled"
        }
    }
)


#1. 状態を定義
class OverAllState(TypedDict):
    topic: str
    poem: str
    joke: str
    content_type: str


#2. ノードを定義
def node_a(state: OverAllState) -> OverAllState:
    poem = model.invoke([f"{state['topic']}をテーマにした詩を書いてください"]).content
    return {
        "poem": poem
    }


def node_b(state: OverAllState) -> OverAllState:
    joke = model.invoke([f"{state['topic']}をテーマにしたジョークを書いてください"]).content
    return {
        "joke": joke
    }


def audit_node(state: OverAllState) -> OverAllState:
    logger.info(
        f"タスクフェーズがすべて実行完了しました。詩は{'生成済み' if state['poem'] else '未生成'}、ジョークは{'生成済み' if state['joke'] else '未生成'}")


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a", node_a)
builder.add_node("node_b", node_b)
builder.add_node("audit_node", audit_node, defer=True)

builder.add_edge(START, "node_a")
builder.add_edge(START, "node_b")
builder.add_edge(START, "audit_node")
builder.add_edge("node_a", END)
builder.add_edge("node_b", END)
builder.add_edge("audit_node", END)

graph = builder.compile()
res = graph.invoke({"topic": "猫"})
print(res)

from IPython.display import display

display(graph)
